# 因果注意力

上一章，我们实现了自注意力机制，词元之间终于可以看到彼此。但是我们也指出了一个致命漏洞：每个词元都可以看到所有位置，包括它后面的词元。

这意味着什么？

假设模型正在处理第 3 个词元，准备预测第 4 个词元。这时它不仅能看到第 1、2 个词元，还能看到第 4、5、6 个词元，也就是答案本身。可以想象训练时模型轻松作弊，损失值很漂亮；但是推理时答案根本还不存在，模型立即露馅。

这就像一个学生写作业偷看标准答案，平时成绩很好。但是到了考场上没有答案可抄，就会原型毕露。

## 因果掩码

解决办法其实很直接：把未来的信息遮盖起来。

具体来说，在计算出注意力分数矩阵之后，把矩阵中所有**未来位置**的数值替代成一个极大的负数（比如 $-10^9$）。这样经过 Softmax 之后，这些位置的注意力权重就趋近于 0，相当于被完全忽略，不参与预测。

这种处理方式称为**因果掩码**（Causal Mask）：每个位置只能看到自己和之前的位置，不能看到之后的位置。这正好符合语言生成的因果关系，下一个词只能由前面的词决定。

### 下三角矩阵

如何实现因果掩码？

观察注意力分数矩阵的格式：`(上下文尺寸, 上下文尺寸)`。第 $i$ 行第 $j$ 列表示第 $i$ 个词元相对于第 $j$ 个词元的相关程度：

* 当 $j \leq i$ 时：第 $j$ 个词元在第 $i$ 个词元之前，或者就是本身，应该保留；
* 当 $j > i$ 时：第 $j$ 个词元在第 $i$ 个词元之后，应该屏蔽。

也就是说，我们只需要保留矩阵的对角线和对角线以下的下三角形部分，把对角线以上的上三角形部分全部替代成 $-10^9$。NumPy 提供了现成的下三角函数 np.tril()，可以生成一个下三角为 1、上三角为 0 的矩阵，可以直接用它作为掩码。

### 因果注意力

使用因果掩码后的自注意力，称为**因果注意力**（Causal Attention）、或者**掩码自注意力**（Masked Self Attention）。这是 GPT 这类**解码器**（Decoder Only）架构的标准配置。

In [1]:
from pathlib import Path

import requests

from src.core import *

In [2]:
np.random.seed(42)

## 数据集

### 字符编码数据集

In [3]:
class CharDataset(Dataset):

    def __init__(self, filename, batch_size=1, context_size=64, stride=None, split=0.9):
        self.filename = filename
        self.context_size = context_size
        self.stride = stride if stride is not None else context_size // 2
        self.split = split
        super().__init__(batch_size)

    def load(self):
        with open(self.filename, encoding="utf-8") as f:
            text = f.read()

        self.vocab = sorted(set(text))
        self.vocab_size = len(self.vocab)
        self.stoi = {ch: i for i, ch in enumerate(self.vocab)}
        self.itos = {i: ch for i, ch in enumerate(self.vocab)}
        self.tokens = self.encode(text)

        split = int(len(self.tokens) * self.split)
        self.train_data = self._pack(self.tokens[:split])
        self.test_data = self._pack(self.tokens[split:])

    def _pack(self, tokens):
        x, y = [], []
        for i in range(0, len(tokens) - self.context_size - 1, self.stride):
            x.append(tokens[i: i + self.context_size])
            y.append(tokens[i + 1: i + self.context_size + 1])
        return x, y

    def encode(self, symbols):
        return [self.stoi[s] for s in symbols]

    def decode(self, tokens):
        return "".join(self.itos[t] for t in tokens)

## 下三角矩阵层

我们将因果掩码设计成单独的**下三角矩阵层**（Tril Lower Layer）：

* **前向传播**：用 np.tril() 生成下三角掩码，把上三角位置的数值替代成 $-10^9$；
* **反向传播**：梯度只通过下三角位置回传。上三角位置没有参与前向传播，所以梯度为 0。

In [5]:
class Tril(Layer):

    def __init__(self, value=-1e9):
        super().__init__()
        self.value = value

    def forward(self, x: Tensor):
        keep = np.tril(np.ones(x.shape[-2:]))
        p = Tensor(np.where(keep, x.data, self.value))

        def gradient_fn():
            x.grad += p.grad * keep

        return p.attach(gradient_fn, {x})

## 激活函数

### 高斯误差线性单元激活函数（GELU）

In [4]:
class GELU(Layer):

    def __init__(self):
        super().__init__()
        self.c = np.sqrt(2.0 / np.pi)

    def forward(self, x: Tensor):
        tanh = np.tanh(self.c * (x.data + 0.044715 * x.data ** 3))
        a = Tensor(0.5 * x.data * (1.0 + tanh))

        def gradient_fn():
            grad = 0.5 * (1.0 + tanh) + 0.5 * x.data * (1.0 - tanh ** 2) * self.c * (1.0 + 3.0 * 0.044715 * x.data ** 2)
            x.grad += a.grad * grad

        return a.attach(gradient_fn, parents={x})

## 损失函数

### GPT 交叉熵损失函数

In [6]:
class GPTLoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        exp = np.exp(p.data - np.max(p.data, axis=-1, keepdims=True))
        softmax = exp / np.sum(exp, axis=-1, keepdims=True)

        flat_softmax = softmax.reshape(-1, softmax.shape[-1])
        flat_y = y.data.reshape(-1).astype(np.int64)
        n = len(flat_y)
        rows = np.arange(n)

        log = np.log(np.clip(flat_softmax[rows, flat_y], 1e-10, 1))
        ce = Tensor(0 - np.sum(log) / n)

        def gradient_fn():
            flat_grad = flat_softmax.copy()
            flat_grad[rows, flat_y] -= 1
            p.grad += ce.grad * flat_grad.reshape(softmax.shape) / n

        return ce.attach(gradient_fn, parents={p})

## 模型

### GPT 嵌入层

In [7]:
class GPTEmbedding(Composite):

    def __init__(self, vocab_size, context_size, embedding_size):
        self.embedding = Embedding(vocab_size, embedding_size)
        self.positional_embedding = Embedding(context_size, embedding_size)

        super().__init__([self.embedding,
                          self.positional_embedding])

    def forward(self, x: Tensor):
        token = self.embedding(x)
        position = self.positional_embedding(Tensor(range(x.shape[1])))
        return token + position

### GPT 注意力层

在 GPT 注意力层，我们使用下三角矩阵 mask 屏蔽掉注意力分数矩阵 scores 的未来位置。

```python
weights = self.softmax(self.mask(scores))
```

In [8]:
class GPTAttention(Composite):

    def __init__(self, embedding_size):
        self.embedding_size = embedding_size

        self.mask = Tril()
        self.softmax = Softmax()
        self.output = Linear(embedding_size, embedding_size)

        super().__init__([self.mask,
                          self.softmax,
                          self.output])

    def forward(self, x: Tensor):
        scale = Tensor(np.array(1.0 / np.sqrt(self.embedding_size)))
        scores = x @ x.transpose((0, 2, 1)) * scale
        weights = self.softmax(self.mask(scores))
        return self.output(weights @ x)

### GPT 前馈网络层

In [9]:
class GPTFeedForward(Composite):

    def __init__(self, embedding_size):
        self.input = Linear(embedding_size, embedding_size * 4)
        self.gelu = GELU()
        self.output = Linear(embedding_size * 4, embedding_size)

        super().__init__([self.input,
                          self.gelu,
                          self.output])

    def forward(self, x: Tensor):
        h = self.gelu(self.input(x))
        return self.output(h)

### GPT 转换器模块

In [10]:
class GPTTransformer(Composite):

    def __init__(self, embedding_size):
        self.attention = GPTAttention(embedding_size)
        self.feed_forward = GPTFeedForward(embedding_size)

        super().__init__([self.attention,
                          self.feed_forward])

    def forward(self, x: Tensor):
        x = self.attention(x)
        return self.feed_forward(x)

### GPT 输出层

In [11]:
class GPTOutput(Composite):

    def __init__(self, embedding_size, vocab_size):
        self.output = Linear(embedding_size, vocab_size)

        super().__init__([self.output])

    def forward(self, x: Tensor):
        return self.output(x)

### GPT 模型

In [12]:
class GPT(Composite):

    def __init__(self, vocab_size, context_size, embedding_size, blocks):
        self.embedding = GPTEmbedding(vocab_size, context_size, embedding_size)
        self.transformers = [GPTTransformer(embedding_size) for _ in range(blocks)]
        self.output = GPTOutput(embedding_size, vocab_size)

        super().__init__([self.embedding] + self.transformers + [self.output])

    def forward(self, x: Tensor):
        x = self.embedding(x)
        for layer in self.transformers:
            x = layer(x)
        return self.output(x)

## GPT 训练器

In [13]:
class GPTTrainer(Trainer):

    def train(self, dataset, epochs, scheduler=None):
        dataset.train()
        self.model.train()

        steps = 0
        for epoch in range(epochs):
            for i in range(len(dataset)):
                if scheduler is not None:
                    self.optimizer.lr = scheduler.step(steps)

                feature, label = dataset[i]

                self.optimizer.zero_grad()
                prediction = self.model(feature)
                loss = self.loss_fn(prediction, label)
                loss.backward()
                self.optimizer.clip_grad_norm()
                self.optimizer.step()
                steps += 1

    def test(self, dataset):
        dataset.eval()
        self.model.eval()

        predictions = []
        total_loss = 0.0
        with Tensor.no_grad():
            for i in range(len(dataset)):
                feature, label = dataset[i]
                prediction = self.model(feature)
                loss = self.loss_fn(prediction, label)
                predictions.append(prediction)
                total_loss += float(loss.data)

        dataset.train()
        self.model.train()
        return predictions, total_loss / len(dataset)

    def generate(self, dataset, prompt, steps=512):
        self.model.eval()
        tokens = dataset.encode(prompt)

        with Tensor.no_grad():
            for _ in range(steps):
                feature = Tensor([tokens[-dataset.context_size:]])
                logits = self.model(feature)

                last_logits = logits.data[0, -1]
                exp = np.exp(last_logits - np.max(last_logits))
                probs = exp / np.sum(exp)
                token = np.random.choice(len(probs), p=probs)
                tokens.append(token)

        return dataset.decode(tokens)

## 超参数

### 学习率

In [15]:
LEARNING_RATE = 0.001

### 批大小

In [ ]:
BATCH_SIZE = 4

### 轮数

In [ ]:
EPOCHS = 2

### 上下文尺寸

In [ ]:
CONTEXT_SIZE = 32

### 嵌入尺寸

In [ ]:
EMBEDDING_SIZE = 64

### 转换器模块数

In [ ]:
BLOCKS = 2

## 数据准备

In [16]:
DATA_FILE = "../../../tinyshakespeare.txt"

file = Path(DATA_FILE)
if not file.exists():
    file.parent.mkdir(parents=True, exist_ok=True)
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    response = requests.get(url)
    response.raise_for_status()
    file.write_text(response.text)

## 建模

In [17]:
dataset = CharDataset(DATA_FILE, BATCH_SIZE, CONTEXT_SIZE)
model = GPT(dataset.vocab_size, CONTEXT_SIZE, EMBEDDING_SIZE, BLOCKS)
loss_fn = GPTLoss()
optimizer = AdamWOptimizer(model.parameters, lr=LEARNING_RATE)
trainer = GPTTrainer(model, loss_fn, optimizer)

## 训练

In [18]:
scheduler = WarmupCosineScheduler(LEARNING_RATE, EPOCHS * len(dataset), 100, LEARNING_RATE / 10)
trainer.train(dataset, EPOCHS, scheduler)

## 评估

In [19]:
prediction, loss = trainer.test(dataset)
print(f'prediction:\t{len(prediction)} steps, each {prediction[0].data.shape}\nloss:\t{loss}')

prediction:	1742 steps, each (4, 32, 65)
loss:	0.22664792747924414


## 推理

In [20]:
print(trainer.generate(dataset, prompt="ROMEO:"))

ROMEO:
Famtany at!--will and, though this him at.

ISTARIO:
Hen ifthat not mid'dses mepion.

LERMIONA:
Sordecer, wor I tontlouce alsucitend waseres, It what you sor ande bust trune do's to whave with Me
In keal dies the is my antas;
I fill wak selse that of this, a llair, sir, ande not ure.

NANGETABELLA:
And Gicacer sennot
He Govely fo notell, so your her that laly.
Her my woo,
For your her her themest, sir, sir?

TORALY:
Norg lo dong of Werlowes ant mee. brenest.

TROO:
The im mepoungel mage ather.

POMEMIO:
Th


本章，我们用**因果掩码**堵上了自注意力**偷看答案**的漏洞。至此，自注意力机制的核心已经完整。这从推理结果可以看出，我们已经可以从推理结果中看到明显的句式和戏剧结构。

## 课后练习

思考一下为什么下三角矩阵层里要用 $-10^9$ 来填充上三角的未来位置？试试把它改成 0，观察一下训练效果有什么变化？